In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [ ]:
!pip install datasets
from datasets import load_dataset

dataset = load_dataset("imdb")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [ ]:
dataset["train"]
dataset["test"]

Dataset({
    features: ['text', 'label'],
    num_rows: 25000
})

In [ ]:
import re

def tokenize(text):
    text = text.lower()
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"[^a-zA-Z]", " ", text)
    tokens = text.split()
    return tokens

In [ ]:
print(tokenize(dataset["train"][0]["text"])[:20])

['i', 'rented', 'i', 'am', 'curious', 'yellow', 'from', 'my', 'video', 'store', 'because', 'of', 'all', 'the', 'controversy', 'that', 'surrounded', 'it', 'when', 'it']


In [ ]:
from collections import Counter

counter = Counter()

for sample in dataset["train"]:
    tokens = tokenize(sample["text"])
    counter.update(tokens)

In [ ]:
MAX_VOCAB_SIZE = 20000
MAX_LEN = 200

# Build vocabulary
most_common = counter.most_common(MAX_VOCAB_SIZE)

vocab = {"<PAD>": 0, "<UNK>": 1}

for idx, (word, _) in enumerate(most_common):
    vocab[word] = idx + 2

vocab_size = len(vocab)
print("Vocab size:", vocab_size)

# Encoding function
def encode(text):
    tokens = tokenize(text)

    ids = [vocab.get(word, vocab["<UNK>"]) for word in tokens]

    if len(ids) > MAX_LEN:
        ids = ids[:MAX_LEN]
    else:
        ids += [vocab["<PAD>"]] * (MAX_LEN - len(ids))

    return torch.tensor(ids, dtype=torch.long)

Vocab size: 20002


In [ ]:
vocab_size = len(vocab)
print(vocab_size)

20002


In [ ]:
class IMDBDataset(Dataset):

    def __init__(self, split):
        self.texts = split["text"]
        self.labels = split["label"]

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = encode(self.texts[idx])
        label = torch.tensor(self.labels[idx], dtype=torch.float)
        return text, label

In [ ]:
full_train_dataset = IMDBDataset(dataset["train"])
test_dataset = IMDBDataset(dataset["test"])

In [ ]:
from torch.utils.data import random_split

train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size

train_dataset, val_dataset = random_split(
    full_train_dataset,
    [train_size, val_size]
)

In [ ]:
BATCH_SIZE = 64

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

In [ ]:
batch = next(iter(train_loader))
print(batch[0].shape)
print(batch[1].shape)

torch.Size([64, 200])
torch.Size([64])


In [ ]:
class LSTMClassifier(nn.Module):

    def __init__(self, vocab_size, embed_dim, hidden_dim, dropout=0.5):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        # Use SMALL fixed dropout inside LSTM
        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            num_layers=2,
            batch_first=True,
            dropout=0.2  # fixed small value
        )

        # Tunable dropout only here
        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        embedded = self.embedding(x)
        output, (hidden, cell) = self.lstm(embedded)
        final_hidden = hidden[-1]
        dropped = self.dropout(final_hidden)
        logits = self.fc(dropped)
        return logits

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
def binary_accuracy(preds, labels):
    probs = torch.sigmoid(preds)
    rounded = torch.round(probs)
    correct = (rounded == labels).float()
    return correct.sum() / len(correct)

In [ ]:
def train(model, loader, optimizer, criterion):

    model.train()
    total_loss = 0
    total_acc = 0

    for texts, labels in loader:

        texts = texts.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(texts).squeeze(1)

        loss = criterion(outputs, labels)
        acc = binary_accuracy(outputs, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        total_acc += acc.item()

    return total_loss / len(loader), total_acc / len(loader)

In [ ]:
def evaluate(model, loader, criterion):

    model.eval()
    total_loss = 0
    total_acc = 0

    with torch.no_grad():
        for texts, labels in loader:

            texts = texts.to(device)
            labels = labels.to(device)

            outputs = model(texts).squeeze(1)

            loss = criterion(outputs, labels)
            acc = binary_accuracy(outputs, labels)

            total_loss += loss.item()
            total_acc += acc.item()

    return total_loss / len(loader), total_acc / len(loader)

In [ ]:
!pip install optuna
import optuna

In [ ]:
def objective(trial):

    hidden_dim = trial.suggest_categorical("hidden_dim", [128, 256, 512])
    dropout = trial.suggest_float("dropout", 0.2, 0.6)
    lr = trial.suggest_float("lr", 1e-4, 5e-3, log=True)

    model = LSTMClassifier(
        vocab_size,
        embed_dim=128,
        hidden_dim=hidden_dim,
        dropout=dropout
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()

    EPOCHS = 5

    for epoch in range(EPOCHS):

        train_loss, train_acc = train(model, train_loader, optimizer, criterion)
        val_loss, val_acc = evaluate(model, val_loader, criterion)

        print(f"Trial {trial.number} | Epoch {epoch+1} | Val Acc: {val_acc:.4f}")

        # Report intermediate result to Optuna
        trial.report(val_acc, step=epoch)

        # Prune if needed
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return val_acc

In [ ]:
study = optuna.create_study(
    direction="maximize",
    pruner=optuna.pruners.MedianPruner()
)

study.optimize(objective, n_trials=10)

print("Best trial:")
print(study.best_trial.params)
print("Best validation accuracy:", study.best_trial.value)

[I 2026-03-01 13:55:36,546] A new study created in memory with name: no-name-e23a608a-6f44-49ae-9726-73d71c793c89


Trial 0 | Epoch 1 | Val Acc: 0.5326
Trial 0 | Epoch 2 | Val Acc: 0.6094
Trial 0 | Epoch 3 | Val Acc: 0.6222
Trial 0 | Epoch 4 | Val Acc: 0.5938


[I 2026-03-01 13:57:50,216] Trial 0 finished with value: 0.6212420886075949 and parameters: {'hidden_dim': 256, 'dropout': 0.5367278223321796, 'lr': 0.000279062540329973}. Best is trial 0 with value: 0.6212420886075949.


Trial 0 | Epoch 5 | Val Acc: 0.6212
Trial 1 | Epoch 1 | Val Acc: 0.5006
Trial 1 | Epoch 2 | Val Acc: 0.5006
Trial 1 | Epoch 3 | Val Acc: 0.5006
Trial 1 | Epoch 4 | Val Acc: 0.4994


[I 2026-03-01 13:59:59,852] Trial 1 finished with value: 0.4994066455696203 and parameters: {'hidden_dim': 256, 'dropout': 0.2752996979284315, 'lr': 0.004889983482752403}. Best is trial 0 with value: 0.6212420886075949.


...

Trial 7 | Epoch 1 | Val Acc: 0.5004


[I 2026-03-01 14:09:37,736] Trial 8 pruned. 


Trial 8 | Epoch 1 | Val Acc: 0.5018


[I 2026-03-01 14:10:03,318] Trial 9 pruned. 


Trial 9 | Epoch 1 | Val Acc: 0.5034
Best trial:
{'hidden_dim': 128, 'dropout': 0.200626583708863, 'lr': 0.0021637118763473393}
Best validation accuracy: 0.8346518987341772


In [ ]:
best_params = study.best_trial.params

In [ ]:
best_hidden = best_params["hidden_dim"]
best_dropout = best_params["dropout"]
best_lr = best_params["lr"]

model = LSTMClassifier(
    vocab_size,
    embed_dim=128,
    hidden_dim=best_hidden,
    dropout=best_dropout
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=best_lr)
criterion = nn.BCEWithLogitsLoss()

In [ ]:
EPOCHS = 10

for epoch in range(EPOCHS):
    train_loss, train_acc = train(model, train_loader, optimizer, criterion)
    val_loss, val_acc = evaluate(model, val_loader, criterion)
    print(f"Epoch {epoch+1} | Val Acc: {val_acc:.4f}")

Epoch 1 | Val Acc: 0.5105
Epoch 2 | Val Acc: 0.5564
Epoch 3 | Val Acc: 0.7518
Epoch 4 | Val Acc: 0.7789
Epoch 5 | Val Acc: 0.8370
Epoch 6 | Val Acc: 0.8432
Epoch 7 | Val Acc: 0.8378
Epoch 8 | Val Acc: 0.8376
Epoch 9 | Val Acc: 0.8465
Epoch 10 | Val Acc: 0.8463


In [ ]:
torch.save(model.state_dict(), "best_lstm_model.pt")
from google.colab import files; files.download("best_lstm_model.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
test_loss, test_acc = evaluate(model, test_loader, criterion)
print("Test Accuracy:", test_acc)

Test Accuracy: 0.8098145779746267
